In [ ]:
import glob
import os
from pathlib import Path

import autoroot
import hydra
import matplotlib.pyplot as plt
import numpy as np
import scipy
import torch
from omegaconf import OmegaConf

from src.data.lf.lf_datamodule import LF
from src.modules.lit_module import LitModule
from src.modules.models.sequential import SequentialModel
from src.utils import initialize_resolvers
from src.utils.tools import get_rho_lstsq

initialize_resolvers()

In [ ]:
def data_illustration(inputs, outputs):

    fig, ax = plt.subplots(1, 2, figsize=(15, 5))

    ax[0].plot(inputs[0, :, 0].numpy())
    ax[0].plot(outputs[0, :, 0].numpy())

    ax[1].plot(get_rho_lstsq(inputs, outputs, end_point=False)[:, 0])

In [ ]:
data1 = LF(
    data_dir="",
    N=50,
    T=1024,
    D=1,
    rho_name="impulse",
    rho_param={"delta": 64},
    input_generator="gaussian",
).dataset_construction()

data_illustration(data1[:][0], data1[:][1])

In [ ]:
path = Path("./logs/train/2024-05-09_00-52-04")
ckpt = max((path / "checkpoints").glob("*.ckpt"), key=os.path.getctime)

ckpt_init = path / "checkpoints" / "initial_checkpoint.ckpt"
model_init = LitModule.load_from_checkpoint(ckpt_init)
model = LitModule.load_from_checkpoint(ckpt)
x = data1[:][0]
y = data1[:][1]
yhat = model(x.to("cuda:0")).cpu().detach()
data_illustration(data1[:][0], yhat)

In [ ]:
A_init = model_init.model.layers[0].A

In [ ]:
At = model.model.layers[0].A
Bt = model.model.layers[0].B
Ct = model.model.layers[0].C

A = model.model.layers[0].A.data.cpu().numpy()
B = model.model.layers[0].B.data.cpu().numpy()
C = model.model.layers[0].C.data.cpu().numpy()

In [ ]:
plt.plot([(C @ np.linalg.matrix_power(A, i) @ B).item() for i in range(325)])

In [ ]:
eigs = np.linalg.eigvals(np.linalg.matrix_power(A, 1))
plt.scatter(eigs.real, eigs.imag, s=1)
plt.gca().set_aspect("equal", adjustable="box")

In [ ]:
K = model.model.layers[0]._K_conv(torch.tensor(A), torch.tensor(B), torch.tensor(C), 128)

In [ ]:
torch.linalg.qr(torch.ones(4, 4, requires_grad=True)).Q

In [ ]:
inputs = torch.rand(1, 128, 1)
output = model.model.layers[0]._non_circular_convolution(inputs, K)

In [ ]:
def _make_hippo(N: int) -> np.ndarray:
    def idx2value(n: int, k: int):
        if n > k:
            return np.sqrt(2 * n + 1) * np.sqrt(2 * k + 1)
        elif n == k:
            return n + 1
        else:
            return 0

    hippo = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            hippo[i, j] = idx2value(i + 1, j + 1)
    return hippo

In [ ]:
_make_hippo(5)

In [ ]:
import autoroot
import numpy as np
import torch

from src.modules.layers.ssm import scanSSM

In [ ]:
model = scanSSM(1, 1, 16)
gen1 = model._K_gen_simple(model.A, model.B, model.C, 5)
gen2 = model._K_gen_inverse(model.A, model.B, model.C, 5)

model._conv_from_gen(gen1, 5)

In [ ]:
model._conv_from_gen(gen2, 5)

In [ ]:
model._K_conv(model.A, model.B, model.C, 5)

In [ ]:
z = torch.tensor(np.exp((-2j * np.pi) * (np.arange(5) / 5)))
torch.vmap(gen1)(z), torch.vmap(gen2)(z)

In [ ]:
z[0]

In [ ]:
gen2(z[0])

In [ ]:
gen1(z[0])